In [ ]:
import pathlib, os, sys

# Pastikan notebook dijalankan dari folder notebooks/
NOTEBOOK_DIR = pathlib.Path.cwd()
if not (NOTEBOOK_DIR / '..' / 'data').exists():
    raise RuntimeError("Jalankan notebook ini dari folder notebooks/")

PROJECT_ROOT = (NOTEBOOK_DIR / '..').resolve()
DATA_DIR = PROJECT_ROOT / 'data'
DETECTOR_DATASET_DIR = DATA_DIR / 'detector_dataset'
MODEL_DIR = PROJECT_ROOT / 'models'
OUTPUT_DIR = PROJECT_ROOT / 'output'

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset detektor: {DETECTOR_DATASET_DIR}")


In [ ]:
# Verifikasi dataset dan perbarui data.yaml dengan path absolut
yaml_path = DETECTOR_DATASET_DIR / 'data.yaml'
if not yaml_path.exists():
    raise FileNotFoundError(f"data.yaml tidak ditemukan di {DETECTOR_DATASET_DIR}")

# Tulis ulang data.yaml dengan path absolut
yaml_content = f"""train: {DETECTOR_DATASET_DIR / 'images' / 'train'}
val: {DETECTOR_DATASET_DIR / 'images' / 'val'}
nc: 1
names: ['medicine_package']
"""
with open(yaml_path, 'w') as f:
    f.write(yaml_content)
print("data.yaml telah diperbarui dengan path absolut:")
print(yaml_content)

# Konfigurasi training
EPOCHS = 50
IMG_SIZE = 640
BATCH_SIZE = 4   # kecil untuk CPU
WORKERS = 2


In [ ]:
# Opsional: hapus folder hasil training sebelumnya untuk memulai bersih
import shutil
train_output = OUTPUT_DIR / 'detector_train'
if train_output.exists():
    shutil.rmtree(train_output)
    print(f"Folder {train_output} dihapus.")
else:
    print("Tidak ada folder training lama.")


In [ ]:
from ultralytics import YOLO

# Load model nano pretrained
model = YOLO('yolov8n.pt')

results = model.train(
    data=str(yaml_path),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    workers=WORKERS,
    device='cpu',
    project=str(OUTPUT_DIR),
    name='detector_train',
    exist_ok=True,
    verbose=True
)


In [ ]:
# Evaluasi model terbaik
best_model_path = OUTPUT_DIR / 'detector_train' / 'weights' / 'best.pt'
model = YOLO(str(best_model_path))

metrics = model.val(data=str(yaml_path), device='cpu')
print(metrics)


In [ ]:
# Ekspor model final
final_model_path = MODEL_DIR / 'panadol_detector.pt'
model.save(str(final_model_path))
print(f"Model detektor disimpan: {final_model_path}")


In [ ]:
# Uji penolakan non‑obat (gunakan beberapa gambar uji)
# Daftar gambar uji – sesuaikan dengan file yang tersedia di demo_test atau dataset
test_images = [
    DATA_DIR / 'demo_test' / 'obat_asli.jpg',
    DATA_DIR / 'demo_test' / 'obat_palsu.jpg',
    DATA_DIR / 'demo_test' / 'buku.jpg',
    DATA_DIR / 'demo_test' / 'meja.jpg',
    DATA_DIR / 'demo_test' / 'random.jpg',
]

model = YOLO(str(final_model_path))

for img_path in test_images:
    if not img_path.exists():
        print(f"File tidak ditemukan: {img_path}")
        continue
    results = model(str(img_path), device='cpu')
    detections = results[0].boxes
    if detections is None or len(detections) == 0:
        print(f"{img_path.name}: TIDAK TERDETEKSI -> DITOLAK ✅")
    else:
        confs = detections.conf.tolist()
        print(f"{img_path.name}: TERDETEKSI (confidence: {max(confs):.2f}) ⚠️")
